# Question 18: Implement Speculative Decoding (SOLUTION)

### Difficulty: Hard

### Problem Statement

Autoregressive decoding is slow because each token requires a full forward pass through the model. **Speculative decoding** speeds this up by using a small, fast **draft model** to propose K tokens, then verifying all K tokens in a **single** forward pass of the larger **target model**.

The key insight is that the target model can score all K proposed tokens in parallel (since we know the draft tokens), and we use **rejection sampling** to decide which draft tokens to accept. This guarantees that the output distribution matches exactly what the target model would produce with standard autoregressive decoding.

Your task is to:
1. Create a small `DraftModel` and a larger `TargetModel`.
2. Implement the speculative decoding algorithm with rejection sampling.
3. Compare speculative decoding to standard (target-only) decoding.

---

### Requirements

1. **`DraftModel`**: A small language model (e.g., 1-layer with small hidden dim) that maps `(batch_size, seq_len, vocab_size)` logits.

2. **`TargetModel`**: A larger language model (e.g., 3-layer with larger hidden dim) that produces the "true" distribution.

3. **`speculative_decode(draft_model, target_model, input_ids, K=5, ...)`**:
   - Use the draft model to autoregressively generate K candidate tokens.
   - Run the target model on the full sequence (input + K draft tokens) in one forward pass.
   - For each draft token, compare `p_target(token)` vs `p_draft(token)`. Accept with probability `min(1, p_target / p_draft)`. On rejection, resample from a corrected distribution.
   - Return the accepted tokens.

4. **Validation**: Show the acceptance rate across multiple runs. Compare wall-clock time of speculative vs standard decoding.

---

### Constraints

- Use only PyTorch operations.
- The rejection sampling must be mathematically correct: on rejection at position i, sample from `max(0, p_target - p_draft)` normalized.
- Both models share the same vocabulary size.

---

### Company Tags

Google, DeepMind, Anthropic, Apple

---

<details>
  <summary>Hint</summary>

  - The draft model generates K tokens one at a time, storing both the tokens and the draft probabilities at each step.
  - The target model scores all K+1 positions in a single forward pass (input + K draft tokens). The +1 is for the token after the last draft token.
  - For rejection sampling: draw `r ~ Uniform(0, 1)`. Accept draft token `x` if `r < p_target(x) / p_draft(x)`. Otherwise, sample from `normalize(max(0, p_target - p_draft))`.
  - If all K tokens are accepted, you get a bonus: sample one more token from the target model's distribution at position K+1.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

In [ ]:
# Configuration
torch.manual_seed(42)

vocab_size = 128
d_model_draft = 64
d_model_target = 256
max_seq_len = 64
K = 5  # Number of speculative tokens

device = "cpu"
print(f"Vocab size: {vocab_size}, Draft dim: {d_model_draft}, Target dim: {d_model_target}, K: {K}")

In [ ]:
class DraftModel(nn.Module):
    """
    Small, fast model that proposes candidate tokens.
    Simple architecture: Embedding -> Linear -> ReLU -> Linear -> logits
    """
    
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.fc1 = nn.Linear(d_model, d_model)
        self.fc2 = nn.Linear(d_model, vocab_size)
    
    def forward(self, input_ids):
        """
        Args:
            input_ids: (batch_size, seq_len) token indices
        Returns:
            logits: (batch_size, seq_len, vocab_size)
        """
        x = self.embedding(input_ids)       # (B, S, d_model)
        x = F.relu(self.fc1(x))             # (B, S, d_model)
        logits = self.fc2(x)                # (B, S, vocab_size)
        return logits


class TargetModel(nn.Module):
    """
    Larger, slower model that produces the "true" distribution.
    Architecture: Embedding -> 3x(Linear -> ReLU) -> Linear -> logits
    """
    
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.fc1 = nn.Linear(d_model, d_model)
        self.fc2 = nn.Linear(d_model, d_model)
        self.fc3 = nn.Linear(d_model, d_model)
        self.fc_out = nn.Linear(d_model, vocab_size)
    
    def forward(self, input_ids):
        """
        Args:
            input_ids: (batch_size, seq_len) token indices
        Returns:
            logits: (batch_size, seq_len, vocab_size)
        """
        x = self.embedding(input_ids)       # (B, S, d_model)
        x = F.relu(self.fc1(x))             # (B, S, d_model)
        x = F.relu(self.fc2(x))             # (B, S, d_model)
        x = F.relu(self.fc3(x))             # (B, S, d_model)
        logits = self.fc_out(x)             # (B, S, vocab_size)
        return logits


print("Model classes defined.")

In [ ]:
def standard_decode(model, input_ids, num_tokens, temperature=1.0):
    """
    Standard autoregressive decoding from the target model.
    Generates one token at a time.
    
    Args:
        model: The target model
        input_ids: (1, prompt_len) initial token indices
        num_tokens: Number of tokens to generate
        temperature: Sampling temperature
        
    Returns:
        generated_ids: (1, prompt_len + num_tokens) full sequence
    """
    current_ids = input_ids.clone()
    
    for _ in range(num_tokens):
        logits = model(current_ids)                         # (1, seq_len, vocab_size)
        next_logits = logits[:, -1, :] / temperature        # (1, vocab_size)
        probs = F.softmax(next_logits, dim=-1)              # (1, vocab_size)
        next_token = torch.multinomial(probs, num_samples=1)  # (1, 1)
        current_ids = torch.cat([current_ids, next_token], dim=1)
    
    return current_ids


def speculative_decode(draft_model, target_model, input_ids, K=5, temperature=1.0):
    """
    Speculative decoding: draft model proposes K tokens, target model verifies.
    
    Args:
        draft_model: Small, fast model for proposing tokens
        target_model: Large, accurate model for verification
        input_ids: (1, prompt_len) initial token indices
        K: Number of speculative tokens to propose
        temperature: Sampling temperature
        
    Returns:
        accepted_tokens: List of accepted token ids
        num_accepted: Number of draft tokens accepted (out of K)
    """
    # ============================
    # Step 1: Draft phase
    # Generate K tokens from the draft model autoregressively
    # ============================
    draft_tokens = []     # The K proposed tokens
    draft_probs = []      # The draft model's probability for each proposed token
    current_ids = input_ids.clone()
    
    for _ in range(K):
        logits = draft_model(current_ids)                    # (1, seq_len, vocab_size)
        next_logits = logits[:, -1, :] / temperature         # (1, vocab_size)
        probs = F.softmax(next_logits, dim=-1)               # (1, vocab_size)
        next_token = torch.multinomial(probs, num_samples=1) # (1, 1)
        
        draft_tokens.append(next_token.item())
        draft_probs.append(probs.squeeze(0))  # (vocab_size,)
        
        current_ids = torch.cat([current_ids, next_token], dim=1)
    
    # ============================
    # Step 2: Verification phase
    # Run target model on the full sequence (prompt + K draft tokens) in one pass
    # ============================
    # current_ids is (1, prompt_len + K)
    target_logits = target_model(current_ids)  # (1, prompt_len + K, vocab_size)
    
    prompt_len = input_ids.shape[1]
    # Target probabilities at each draft position:
    # Position prompt_len-1 predicts token at prompt_len (first draft token)
    # Position prompt_len predicts token at prompt_len+1 (second draft token)
    # etc.
    
    # ============================
    # Step 3: Rejection sampling
    # ============================
    accepted_tokens = []
    num_accepted = 0
    
    for i in range(K):
        draft_token = draft_tokens[i]
        q_probs = draft_probs[i]  # Draft distribution (vocab_size,)
        
        # Target distribution at this position
        target_pos = prompt_len - 1 + i  # Index into target_logits
        p_logits = target_logits[0, target_pos, :] / temperature
        p_probs = F.softmax(p_logits, dim=-1)  # (vocab_size,)
        
        # Rejection criterion
        p_token = p_probs[draft_token].item()
        q_token = q_probs[draft_token].item()
        
        r = torch.rand(1).item()
        
        if r < min(1.0, p_token / max(q_token, 1e-10)):
            # ACCEPT the draft token
            accepted_tokens.append(draft_token)
            num_accepted += 1
        else:
            # REJECT: sample from the corrected distribution
            # corrected = normalize(max(0, p_target - p_draft))
            corrected = torch.clamp(p_probs - q_probs, min=0)
            corrected_sum = corrected.sum()
            if corrected_sum > 0:
                corrected = corrected / corrected_sum
            else:
                # Fallback to target distribution
                corrected = p_probs
            
            replacement = torch.multinomial(corrected.unsqueeze(0), num_samples=1).item()
            accepted_tokens.append(replacement)
            return accepted_tokens, num_accepted
    
    # ============================
    # Step 4: All K tokens accepted -- sample one bonus token
    # ============================
    bonus_logits = target_logits[0, prompt_len + K - 1, :] / temperature
    bonus_probs = F.softmax(bonus_logits, dim=-1)
    bonus_token = torch.multinomial(bonus_probs.unsqueeze(0), num_samples=1).item()
    accepted_tokens.append(bonus_token)
    
    return accepted_tokens, num_accepted


print("Decoding functions defined.")

In [ ]:
# ============================================================
# Validation & Benchmarking
# ============================================================

torch.manual_seed(42)
draft_model = DraftModel(vocab_size, d_model_draft).to(device).eval()
target_model = TargetModel(vocab_size, d_model_target).to(device).eval()

print(f"Draft model params:  {sum(p.numel() for p in draft_model.parameters()):,}")
print(f"Target model params: {sum(p.numel() for p in target_model.parameters()):,}")
print()

# Generate tokens using speculative decoding (multiple rounds)
prompt = torch.randint(0, vocab_size, (1, 5), device=device)
total_generated = 0
total_accepted = 0
num_rounds = 20

current_ids = prompt.clone()
with torch.no_grad():
    for round_idx in range(num_rounds):
        accepted_tokens, num_accepted = speculative_decode(
            draft_model, target_model, current_ids, K=K, temperature=1.0
        )
        total_generated += len(accepted_tokens)
        total_accepted += num_accepted
        # Append accepted tokens to the sequence
        new_tokens = torch.tensor([accepted_tokens], device=device)
        current_ids = torch.cat([current_ids, new_tokens], dim=1)

acceptance_rate = total_accepted / (num_rounds * K)
print(f"Total tokens generated: {total_generated}")
print(f"Draft tokens accepted: {total_accepted} / {num_rounds * K}")
print(f"Acceptance rate: {acceptance_rate:.1%}")
print(f"Avg tokens per round: {total_generated / num_rounds:.1f} (max possible: {K + 1})")
print()

# --- Benchmark: speculative vs standard ---
num_tokens_to_gen = 50

# Standard decoding
prompt = torch.randint(0, vocab_size, (1, 5), device=device)
start = time.time()
with torch.no_grad():
    std_output = standard_decode(target_model, prompt, num_tokens_to_gen)
std_time = time.time() - start

# Speculative decoding
current_ids = prompt.clone()
spec_count = 0
start = time.time()
with torch.no_grad():
    while spec_count < num_tokens_to_gen:
        accepted_tokens, _ = speculative_decode(draft_model, target_model, current_ids, K=K)
        remaining = num_tokens_to_gen - spec_count
        tokens_to_add = accepted_tokens[:remaining]
        new_tokens = torch.tensor([tokens_to_add], device=device)
        current_ids = torch.cat([current_ids, new_tokens], dim=1)
        spec_count += len(tokens_to_add)
spec_time = time.time() - start

print(f"Standard decoding: {std_time:.4f}s for {num_tokens_to_gen} tokens")
print(f"Speculative decoding: {spec_time:.4f}s for {num_tokens_to_gen} tokens")
print(f"Speedup: {std_time / spec_time:.2f}x")